In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r"D:\Machine Learning\Datasets")  # Raw string for safety

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Load and preprocess data
housing = pd.read_csv("Housing.csv")
dum_hous = pd.get_dummies(housing, drop_first=True)

y = dum_hous[['price']].values          # Keep as numpy for scaling
X = dum_hous.drop('price', axis=1).values

# Scaling
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=25
)

X_scl_trn = scaler_x.fit_transform(X_train)
X_scl_tst = scaler_x.transform(X_test)
y_scl_trn = scaler_y.fit_transform(y_train)
y_scl_tst = scaler_y.transform(y_test)

# Convert to tensors
X_train_tensor = torch.from_numpy(X_scl_trn).float()
y_train_tensor = torch.from_numpy(y_scl_trn).float()
X_test_tensor = torch.from_numpy(X_scl_tst).float()
y_test_tensor = torch.from_numpy(y_scl_tst).float()

# Dataset & DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Good batch size

# Set seed
torch.manual_seed(25)

# === CUSTOM MLP REGRESSOR ===
class MLPRegressor(nn.Module):
    def __init__(self, num_features):
        super(MLPRegressor, self).__init__()  # Critical line
        
        self.linear1 = nn.Linear(in_features=num_features, out_features=8)
        self.linear2 = nn.Linear(in_features=8, out_features=4)
        self.linear3 = nn.Linear(in_features=4, out_features=3)
        self.linear4 = nn.Linear(in_features=3, out_features=1)
        
        self.relu = nn.ReLU()  # Reusable activation

    def forward(self, x):
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.relu(self.linear3(x))
        x = self.linear4(x)      # No activation on final output (regression)
        return x

# Instantiate model
model = MLPRegressor(num_features=X_scl_trn.shape[1])

# Model summary - correct for tabular data
summary(model, input_size=(X_scl_trn.shape[1],))

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Lower LR often better

# Training loop with batches
num_epochs = 1000
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        outputs = model(batch_X)           # Predicted scaled prices
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    if (epoch + 1) % 100 == 0:
        avg_loss = epoch_loss / len(train_loader)
        print(f'Epoch [{epoch+1}/{num_epochs}], Avg Loss: {avg_loss:.6f}')

# Evaluation on test set
model.eval()
with torch.no_grad():
    y_pred_scl = model(X_test_tensor)                # Scaled predictions
    y_pred_scl_np = y_pred_scl.numpy()
    
    # Inverse transform to original price scale
    y_pred = scaler_y.inverse_transform(y_pred_scl_np)
    y_test_original = scaler_y.inverse_transform(y_test)

# R² Score on original scale
r2 = r2_score(y_test_original, y_pred)
print(f"\nTest R² Score: {r2:.4f}")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                    [-1, 8]              96
              ReLU-2                    [-1, 8]               0
            Linear-3                    [-1, 4]              36
              ReLU-4                    [-1, 4]               0
            Linear-5                    [-1, 3]              15
              ReLU-6                    [-1, 3]               0
            Linear-7                    [-1, 1]               4
Total params: 151
Trainable params: 151
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
Epoch [100/1000], Avg Loss: 0.025351
Epoch [200/1000], Avg Loss: 0.025190
Epoch [300/1000], Avg Loss: 0.025293
Epo